# 🧠 SmartTrainer ML: Investigación y Ciencia de Datos

Este cuaderno documenta el proceso de **Ciencia de Datos** detrás del sistema SmartTrainer ML. Aquí exploramos cómo pasamos de una simulación biomecánica a un modelo predictivo de riesgo de lesión.

## 📋 Tabla de Contenidos
1. [Carga de Datos Relacionales](#1.-Carga-de-Datos-Relacionales)
2. [Análisis Exploratorio de Datos (EDA)](#2.-Análisis-Exploratorio-de-Datos-(EDA))
3. [Ingeniería de Variables (Feature Engineering)](#3.-Ingeniería-de-Variables-(Feature-Engineering))
4. [Entrenamiento del Modelo (XGBoost)](#4.-Entrenamiento-del-Modelo-(XGBoost))
5. [Evaluación e Interpretación (SHAP)](#5.-Evaluación-e-Interpretación-(SHAP))

## 1. Carga de Datos Relacionales

El sistema genera los datos en formato relacional (CSV) para evitar redundancia y simular un entorno de base de datos real.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Rutas a los datos
DATA_PATH = "../data/"
users = pd.read_csv(os.path.join(DATA_PATH, "users.csv"))
exercises = pd.read_csv(os.path.join(DATA_PATH, "exercises_catalog.csv"))
training_logs = pd.read_csv(os.path.join(DATA_PATH, "training_logs.csv"))
sessions = pd.read_csv(os.path.join(DATA_PATH, "sessions.csv"))

print(f"Usuarios: {len(users)}")
print(f"Sesiones: {len(sessions)}")
print(f"Logs de Ejercicios: {len(training_logs)}")

## 2. Análisis Exploratorio de Datos (EDA)

### Distrubución de Lesiones
Es vital entender si las clases están balanceadas. En el mundo real, las lesiones son eventos raros.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='injured', data=sessions, palette='viridis')
plt.title("Distribución de la Variable Objetivo (Injured)")
plt.show()

### Correlación entre Fatiga y Lesión
Analizamos cómo la fatiga del Sistema Nervioso Central (SNC) correlaciona con la probabilidad de lesión.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='injured', y='fatigue_snc_total', data=sessions)
plt.title("Fatiga SNC vs Probabilidad de Lesión")
plt.show()

## 3. Ingeniería de Variables (Feature Engineering)

### ¿Por qué estas variables?
1. **Edad/Peso**: Modifican la tolerancia a la carga biomecánica.
2. **Fatiga Acumulada**: El riesgo no viene de un solo set, sino de la acumulación de estrés sistémico.
3. **Dummies de Zonas**: Permiten al modelo identificar riesgos específicos (ej: sobrecarga lumbar por acumular Sentadilla + Peso Muerto).

In [ ]:
# Preparamos el dataset final (JOINs)
df_logs = training_logs.merge(exercises, on='exercise_id')
zones_dummies = df_logs['zonas'].str.get_dummies(sep=',')
df_logs = pd.concat([df_logs, zones_dummies], axis=1)

session_zones = df_logs.groupby('session_id')[zones_dummies.columns].sum()
final_df = sessions.merge(users, on='user_id').merge(session_zones, on='session_id')

final_df.head()

## 4. Entrenamiento del Modelo (XGBoost)

Seleccionamos **XGBoost** por su capacidad de manejar datos tabulares desbalanceados mediante el parámetro `scale_pos_weight`.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

X = final_df.drop(['session_id', 'user_id', 'injured', 'timestamp', 'date_joined'], axis=1, errors='ignore')
y = final_df['injured']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=100, scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1])
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))

## 5. Evaluación e Interpretación (SHAP)

Utilizamos **SHAP** para entender qué variables están empujando la predicción hacia el riesgo.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)


### Conclusiones
- La **fatiga del SNC** es el predictor dominante.
- El historial de lesiones previas tiene un impacto no lineal capturado por el modelo de árboles.
- La sobrecarga en la zona **rodilla** correlaciona fuertemente con eventos de lesión en sesiones coordinadas.